In [13]:
import pandas as pd
import spacy as spc
import scispacy as scspcy
# Multiple ways to get and work with home directory
import os
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from sklearn.cluster import KMeans
import requests
import re
import math
from collections import Counter
from IPython.display import Audio, display
import random
import time
from spacy.matcher import PhraseMatcher

# 1. Using os.path
home_dir = os.path.expanduser('~')
print(f"Home directory (os.path): {home_dir}")

# 2. Using Path from pathlib (modern approach)
home = Path.home()
print(f"Home directory (pathlib): {home}")

# 3. Using environment variables
home_env = os.environ['HOME']  # For Unix/Linux/Mac
# home_env = os.environ['USERPROFILE']  # For Windows
print(f"Home directory (environ): {home_env}")

# Example: List contents of home directory
print("\nContents of home directory:")
#for item in Path.home().iterdir():
#    print(f"- {item.name}")

# Create a new folder (if it doesn't exist)
new_folder = "/Users/naveendeevi/Datasets"
if not os.path.exists(new_folder):
    os.makedirs(new_folder)

# Change directory to the new folder
os.chdir(new_folder)

# Verify current working directory
print(f"Current working directory: {os.getcwd()}")
#files = os.listdir(os.getcwd())
#print(files)

# Step 3: Download and load the model
try:
    nlp = spc.load("en_core_sci_sm")
    
    # Test the model
    text = "Paracetamol is commonly used to treat fever and mild to moderate pain."
    doc = nlp(text)
    
    # Print results
    print("Named Entities:")
    for ent in doc.ents:
        print(f"Entity: {ent.text}, Label: {ent.label_}")
        
except Exception as e:
    print(f"Error: {e}")
    print("\nTroubleshooting steps:")
    print("1. Make sure you're running this in a fresh Python session")
    print("2. Try restarting your Jupyter kernel")
    print("3. Verify your Python environment has write permissions")

Home directory (os.path): /Users/naveendeevi
Home directory (pathlib): /Users/naveendeevi
Home directory (environ): /Users/naveendeevi

Contents of home directory:
Current working directory: /Users/naveendeevi/Datasets
Named Entities:
Entity: Paracetamol, Label: ENTITY
Entity: treat, Label: ENTITY
Entity: fever, Label: ENTITY
Entity: mild, Label: ENTITY
Entity: moderate pain, Label: ENTITY


In [303]:
# Functions
def get_rxcui(drug_name):
    """Get RxCUI from RxNorm for a given drug name."""
    url = f"https://rxnav.nlm.nih.gov/REST/rxcui.json?name={drug_name}"
    print(url)
    response = requests.get(url)
    data = response.json()
    return data.get('idGroup', {}).get('rxnormId', [None])[0]x

def get_drug_classes(rxcui):
    """Get drug classes for a given RxCUI using RxClass API."""
    if not rxcui:
        return None
    #if rxcui.isna():
    #    return None
    if isinstance(rxcui, float) and math.isnan(rxcui):
        return None
    
    rxcui = int(rxcui)
    
    url = f'based '
    
    response = requests.get(url)
    data = response.json()

    classes = []
    for entry in data.get('rxclassDrugInfoList', {}).get('rxclassDrugInfo', []):
        class_name = entry.get('rxclassMinConceptItem', {}).get('className')
        class_type = entry.get('rxclassMinConceptItem', {}).get('classType')
        if class_name and class_type=="ATC1-4": #ATC standard is global standard WHO standard and a little broad
            classes.append(f"{class_name} ({class_type})")  
            break
        elif class_name and class_type=="EPC": #EPC is the FDA standard (US Standard), usually tending more towards on which body part this works on.
            classes.append(f"{class_name} ({class_type})")
    return list(set(classes))  # Deduplicate

def classify_drug(drug_name):
    rxcui = get_rxcui(drug_name)
    if not rxcui:
        #return f"{drug_name}: RxCUI not found"
        return None
    classes = get_drug_classes(rxcui)
    if not classes:
        #return f"{drug_name}: No class info found"
        return None
    return "; ".join(classes)

def play_music(): 
    # List of royalty-free sample audio URLs
    music_urls = [
        "https://www.soundhelix.com/examples/mp3/SoundHelix-Song-1.mp3",
        "https://www.soundhelix.com/examples/mp3/SoundHelix-Song-2.mp3",
        "https://www.soundhelix.com/examples/mp3/SoundHelix-Song-3.mp3"
    ]
    
    # Pick one at random
    selected_url = random.choice(music_urls)
    
    # Trigger point in your logic
    print("🎵 Trigger reached, playing random music from web...")
    
    # Play
    display(Audio(url=selected_url, autoplay=True))

def countdown(t):
    while t:
        mins, secs = divmod(t, 60)
        timer = '{:02d}:{:02d}'.format(mins, secs)
        print(timer, end="\r")
        time.sleep(1)
        t -= 1



SyntaxError: invalid syntax. Perhaps you forgot a comma? (2805219360.py, line 8)

In [298]:
# Import data
df = pd.read_csv('MDD Drugs Input.csv')

### Getting Class

In [284]:
# Get class from RXCUI. Create a dictionary mapping the RXCUI to the class

# Create dictionary 
rxcui_class_dict = {}

# Extract distinct rxcui's & Not nulls
keys = sorted(df[df['rxcui'].notna()]['rxcui'])
j = 0

In [285]:


# Print class from the rxcui and insert into the dictionary
try:
    for i in range(0,len(keys)):
        #print(i)
        print('%d of %d'%(j,len(keys)))
        rxcui_class_dict[int(keys[i])] = get_drug_classes(int(keys[i]))
        if (j>0 and j%100 == 0):
            print("Execution paused for a 3 minutes")
            #time.sleep(180)
            #countdown(180)
            # Code will continue execution after the 5-minute pause
            print("Resumed execution after 3 minutes.")
        j+=1
        # As this code is timing out, a pause of 5 mins is given to the code at 
        #for ratio in (0.00,1.00,0.10):
        #    print(ration)        
except:        
    play_music()
#print(rxcui_class_dict)

0 of 738
1 of 738
2 of 738
3 of 738
4 of 738
5 of 738
6 of 738
7 of 738
8 of 738
9 of 738
10 of 738
11 of 738
12 of 738
13 of 738
14 of 738
15 of 738
16 of 738
17 of 738
18 of 738
19 of 738
20 of 738
21 of 738
22 of 738
23 of 738
24 of 738
25 of 738
26 of 738
27 of 738
28 of 738
29 of 738
30 of 738
31 of 738
32 of 738
33 of 738
34 of 738
35 of 738
36 of 738
37 of 738
38 of 738
39 of 738
40 of 738
41 of 738
42 of 738
43 of 738
44 of 738
45 of 738
46 of 738
47 of 738
48 of 738
49 of 738
50 of 738
51 of 738
52 of 738
53 of 738
54 of 738
55 of 738
56 of 738
57 of 738
58 of 738
59 of 738
60 of 738
🎵 Trigger reached, playing random music from web...


In [46]:
df['class_rxcui'] = df['rxcui'].map(rxcui_class_dict)
#df['Class_rxcui'] = df.loc[:,'rxcui'].apply(lambda x: get_drug_classes(x))

In [283]:
df['rxcui'].map(rxcui_class_dict)

NameError: name 'rxcui_class_dict' is not defined

In [47]:
df.columns

Index(['rxcui', 'medication_name', 'dose', 'ingredient_name',
       'indication_terms_fda', 'class_rxcui'],
      dtype='object')

In [23]:
#mask = df['class_rxcui'].isna()
#df.loc[mask,'medication_name']

keys_med_name = sorted(df[df['class_rxcui'].isna()]['medication_name'])
med_class_dict = {}

j = 0

In [287]:
try:
    for i in range(0,len(keys_med_name)):
        #print(i)
        print('%d of %d'%(j,len(keys_med_name)))
        med_class_dict[keys_med_name[i]] = classify_drug(keys_med_name[i])
        if (j>0 and j%100 == 0):
            print("Execution paused for a 3 minutes")
            #time.sleep(180)
            countdown(180)
            # Code will continue execution after the 5-minute pause
            print("Resumed execution after 3 minutes.")
        j+=1
except: 
    #play_music()
    print(x)

abdominal pain, abnormal dreams, accident, adenocarcinoma, adjustment disorder, adverse reaction, ageusia, agranulocytosis, alanine aminotransferase increased, anorexia, anxiety, arthralgia, asthenia, ataxia, atrioventricular block, back injury, back pain, blepharospasm, blister, blood and lymphatic system disorders, blood glucose increased, blood pressure inadequately controlled, blood pressure increased, breast cyst, bronchitis, cellulitis, cellulitis staphylococcal, cerebrovascular accident, chest pain, chills, chronic insomnia, cold sweat, confusional state, constipation, cough, cyst, decreased appetite, depression, dermatitis, diplopia, disturbance in attention, dizziness, dry mouth, dysgeusia, dyspepsia, dyspnea, dysuria, ear and labyrinth disorders, ear pain, electrocardiogram qrs complex, electrocardiogram qrs complex abnormal, electrocardiogram st-t segment abnormal, elevated mood, enuresis, eosinophilia, erythema, extrasystoles, eye infection, eye pain, eye redness, fall, fat

In [227]:
#med_class_dict

In [48]:
df['class_med_name'] = df['rxcui'].map(med_class_dict)

In [288]:
keys_ing_name_V1 = sorted(df[df['class_rxcui'].isna()]['medication_name'])
keys_ing_name_V2 = sorted(list(set([i.split(' ')[0] for i in keys_ing_name_V1])))
ing_name_class_dict = {}

j = 0
# Print class from the rxcui and insert into the dictionary
try:
    for i in range(0,len(keys_ing_name_V2)):
        #print(i)
        print('%d of %d'%(j,len(keys_ing_name_V2)))
        ing_name_class_dict[keys_ing_name_V2[i]] = classify_drug(keys_ing_name_V2[i])
        if (j>0 and j%100 == 0):
            print("Execution paused for a 3 minutes")
            #time.sleep(180)
            countdown(180)
            # Code will continue execution after the 5-minute pause
            print("Resumed execution after 3 minutes.")
        j+=1
        # As this code is timing out, a pause of 5 mins is given to the code at 
        #for ratio in (0.00,1.00,0.10):
        #    print(ration)  
except:
    play_music()
#print(rxcui_class_dict)
print(ing_name_class_dict)

KeyError: 'class_rxcui'

In [49]:
df['class_med_first_name'] = None
mask = df['class_rxcui'].isna()
df.loc[mask,'class_med_first_name'] = df.loc[mask,'medication_name'].apply(lambda x : x.split(' ')[0]).map(ing_name_class_dict)

In [50]:
#df.shape
#df[df['class_rxcui'].notna()]
df['class_rxcui'] = df['class_rxcui'].apply(lambda x: ';;;'.join(x) if isinstance(x, list) and len(x)>0 else None)

df['coalesced_class_name'] = df[['class_rxcui', 'class_med_name', 'class_med_first_name']].bfill(axis=1).iloc[:, 0]


In [51]:
df.drop(columns=['class_rxcui','class_med_name','class_med_first_name'], inplace=True)

In [52]:
df.loc[df['coalesced_class_name'].isna(),['rxcui','medication_name','ingredient_name','coalesced_class_name']].shape

(14, 4)

In [53]:
set(df.loc[df['coalesced_class_name'].isna(),'medication_name'])

{'12 hr clonidine hydrochloride 0.2 mg extended release oral tablet',
 'buprenorphine-nalox 8-2mg film',
 'dextroamphetamine-amphet er 20 mg cap er 24h',
 'dextroamphetamine-amphet er 5 mg cap er 24h',
 'dextroamphetamine-amphetamine 10 mg tablet',
 'dextroamphetamine-amphetamine 15 mg tablet',
 'dextroamphetamine-amphetamine 20 mg tablet',
 'dextroamphetamine-amphetamine er 10 mg 24hr capsule,extend release',
 'dextroamphetamine-amphetamine er 15 mg 24hr capsule,extend release',
 'dextroamphetamine-amphetamine er 25 mg 24hr capsule,extend release',
 'metoprol suc tab 25mg er',
 'multivitamin, prenatal (prenatal multivitamins with vitamin b complex, vitamin c, minerals and l-methylfolate oral capsule)',
 'nystatin-hydrocortisone-lidocaine-diphenhydramine (magic mouthwash with lidocaine) suspension',
 'zaleplon 10 mg oral capsule [sonata]'}

In [15]:
#df.to_csv('out_MDD.csv')

NameError: name 'df' is not defined

In [195]:
df = pd.read_csv('out_MDD.csv')

In [196]:
df.shape

(972, 7)

In [275]:
df.tail(5)

,Unnamed: 0,rxcui,medication_name,dose,ingredient_name,indication_terms_fda,coalesced_class_name,SE_rxcui,SE_med_nm,SE_ing_nm,Side_Effects_Combined
967,967,NaN,vilazodone hcl 40 mg tablet,NaN,NaN,NaN,Other antidepressants (ATC1-4),NaN,NaN,NaN,
968,968,NaN,vilazodone tab 10mg,NaN,NaN,NaN,Other antidepressants (ATC1-4),NaN,NaN,NaN,
969,969,NaN,zolpidem er 12.5mg tablets,NaN,NaN,NaN,gamma-Aminobutyric Acid A Receptor Positive Mo...,NaN,NaN,NaN,
970,970,NaN,zolpidem tart er 12.5 mg tab,NaN,NaN,NaN,gamma-Aminobutyric Acid A Receptor Positive Mo...,NaN,NaN,NaN,
971,971,NaN,zolpidem tart er 6.25 mg tab,NaN,NaN,NaN,gamma-Aminobutyric Acid A Receptor Positive Mo...,NaN,NaN,NaN,


### The ratio of drugs without class vs with class is 14:972 which is 1.4%. 

## Side Effects

In [198]:
#!python -m spacy download en_core_web_sm


1. Import existing side effects from SIDER (Side Effects) database. Created by Structural and Computational Biology Unit at EMBL Heidelberg, Germany.
2. Import side effects from Spacy library.
3. Union the no.1 & no.2. 
4. Import 'Adverse Reactions', 'Boxed Warning', 'Warnings and Precautions', 'warnings', 'precautions'.
5. Extract side effects from drug and result out the side effects. 


In [239]:
## 🔹 Step 1: Load Side Effect Terms from SIDER
# Make sure to download `meddra_all_se.tsv` from http://sideeffects.embl.de/download/
sider_path = "meddra_all_se.tsv"  # Update path if needed
df_se = pd.read_csv(sider_path, sep='\t', header=None)
df_se.columns = ['stitch_flat', 'stitch_stereo', 'umls_cui', 'side_effect_code', 'method','side_effect_name']
sider_terms = set(df_se['side_effect_name'].str.lower().unique())
sider_terms.update(['suicidal','suicide','gastrointestinal','confusion','libido','weakness','panic attack','aggressiveness',
                   'dizziness','seizures','blurred vision','apnea','restlessness','tremors','hallucinations','delusion',
                   'sleep disturbance','birth defects','miscarriage','falls','factures','change in appetite','euphoria',
                   'backache','behavior','feeding difficulties','dyspnea','birth defects','mortality','nightmares','burning eyes',
                   'oversimulation','poor weight gain','blood dyscrasias','flu','back pain','salvation','twitching','reflexes'])

In [276]:
def get_openfda_ae_terms(drug_name):
    fields_to_try = ["openfda.generic_name", "openfda.brand_name"]
    ae_fields = ['adverse_reactions', 'warnings', 'boxed_warning', 'precautions', 'warnings_and_cautions']
    
    for field in fields_to_try:
        url = f"https://api.fda.gov/drug/label.json?search={field}:\"{drug_name}\"&limit=1"
        print(url)
        response = requests.get(url)
        
        if response.status_code == 200:
            try:
                result = response.json()['results'][0]
                
                extracted_text = []
                for ae_field in ae_fields:
                    val = result.get(ae_field, '')
                    if isinstance(val, list):
                        extracted_text.extend(val)
                    elif isinstance(val, str):
                        extracted_text.append(val)

                combined = ' '.join(extracted_text)
                return [sent.strip().lower() for sent in combined.split('.') if len(sent.strip()) > 3]
            except Exception as e:
                #print(f"Error parsing results: {e}")
                return None
        elif response.status_code != 404:
            #print(f"Request failed with status code {response.status_code}")
            return None
    
    #print("No matching record found.")
    return None

def get_openfda_ae_terms_by_rxcui(rxcui):
    ae_fields = ['adverse_reactions', 'warnings', 'boxed_warning', 'precautions','warnings_and_cautions']

    url = f"https://api.fda.gov/drug/label.json?search=openfda.rxcui:{rxcui}&limit=1"
    response = requests.get(url)

    if response.status_code == 200:
        try:
            result = response.json()['results'][0]

            extracted_text = []
            for ae_field in ae_fields:
                val = result.get(ae_field, '')
                if isinstance(val, list):
                    extracted_text.extend(val)
                elif isinstance(val, str):
                    extracted_text.append(val)

            combined = ' '.join(extracted_text)
            return [sent.strip().lower() for sent in combined.split('.') if len(sent.strip()) > 3]
        except Exception as e:
            #print(f"Error parsing results: {e}")
            return None
    elif response.status_code != 404:
        #print(f"Request failed with status code {response.status_code}")
        return None

    #print("No matching record found.")
    return None


In [241]:
def extract_side_effects_from_text(sider_side_effects_list, combined_text, max_phrase_len=5):
    """
    Extract known side effects from a combined drug text using SIDER terms and SpaCy PhraseMatcher.

    Args:
        sider_side_effects_list (list): List of known side effect phrases (from SIDER or other dictionaries).
        combined_text (str): Drug label text (merged from AE, Warnings, Precautions, etc.).
        max_phrase_len (int): Max length (in words) of phrases to consider. Default is 5.

    Returns:
        List of matched side effect phrases (lowercase, deduplicated).
    """
    # Load SpaCy model
    nlp = spacy.load("en_core_web_sm", disable=["ner", "parser", "lemmatizer"])
    
    # Prepare patterns (only phrases up to max_phrase_len words)
    patterns = [nlp.make_doc(text) for text in sider_side_effects_list if isinstance(text, str) and len(text.split()) <= max_phrase_len]
    
    # Setup matcher
    matcher = PhraseMatcher(nlp.vocab, attr="LOWER")
    matcher.add("SIDE_EFFECT", patterns)

    # Run matcher on text
    doc = nlp(combined_text)
    matches = matcher(doc)
    
    # Extract matched spans
    effect_set = {doc[start:end].text.lower() for match_id, start, end in matches}
    return ', '.join(sorted(effect_set))

In [277]:
#text = get_openfda_ae_terms('doxepin')
x = extract_side_effects_from_text(sider_terms,' '.join(text))

https://api.fda.gov/drug/label.json?search=openfda.generic_name:"doxepin"&limit=1


In [264]:
x

'abdominal pain, abnormal dreams, accident, adenocarcinoma, adjustment disorder, adverse reaction, ageusia, agranulocytosis, alanine aminotransferase increased, anorexia, anxiety, arthralgia, asthenia, ataxia, atrioventricular block, back injury, back pain, blepharospasm, blister, blood and lymphatic system disorders, blood glucose increased, blood pressure inadequately controlled, blood pressure increased, breast cyst, bronchitis, cellulitis, cellulitis staphylococcal, cerebrovascular accident, chest pain, chills, chronic insomnia, cold sweat, confusional state, constipation, cough, cyst, decreased appetite, depression, dermatitis, diplopia, disturbance in attention, dizziness, dry mouth, dysgeusia, dyspepsia, dyspnea, dysuria, ear and labyrinth disorders, ear pain, electrocardiogram qrs complex, electrocardiogram qrs complex abnormal, electrocardiogram st-t segment abnormal, elevated mood, enuresis, eosinophilia, erythema, extrasystoles, eye infection, eye pain, eye redness, fall, fa

In [243]:
# Get side effects from RXCUI. Create a dictionary mapping the RXCUI to the class

# Create dictionary 
rxcui_se_dict = {}

# Extract distinct rxcui's & Not nulls
keys = list(set(sorted(df[df['rxcui'].notna()]['rxcui'])))
j = 0

#print(keys)
#Print class from the rxcui and insert into the dictionary
try:
    for i in range(0,len(keys)):
        #print(keys[i])
        print('%d of %d'%(j,len(keys)),end='\r')
        combined_text = get_openfda_ae_terms_by_rxcui(int(keys[i]))
        #print(combined_text)
        if combined_text != None:
            rxcui_se_dict[int(keys[i])] = extract_side_effects_from_text(sider_terms,' '.join(combined_text))
        #print()
        if (j>0 and j%100 == 0):
            print("Execution paused for a 3 minutes")
            #time.sleep(180)
            countdown(180)
            # Code will continue execution after the 3-minute pause
            print("Resumed execution after 3 minutes.")
        j+=1
except:        
    play_music()
#print(rxcui_class_dict)

Execution paused for a 3 minutes
Resumed execution after 3 minutes.
Execution paused for a 3 minutes
Resumed execution after 3 minutes.
Execution paused for a 3 minutes
Resumed execution after 3 minutes.
335 of 336

In [244]:
df['SE_rxcui'] = df['rxcui'].map(rxcui_se_dict)

In [279]:
# Get side effects from Medication Name. Create a dictionary mapping the RXCUI to the class

# Create dictionary 
med_class_se_dict = {}

# Extract distinct med_names's & Not nulls
med_nm_keys = list(set(sorted(df[df['medication_name'].notna()]['medication_name'])))
j = 0

#print(keys)
#Print class from the rxcui and insert into the dictionary
try:
    for i in range(0,len(med_nm_keys)):
        #print(keys[i])
        print('%d of %d'%(j,len(med_nm_keys)),end='\r')
        combined_text = get_openfda_ae_terms(med_nm_keys[i])
        #print(combined_text)
        if combined_text != None:
            med_class_se_dict[med_nm_keys[i]] = extract_side_effects_from_text(sider_terms,' '.join(combined_text))
        #print()
        if (j>0 and j%100 == 0):
            print("Execution paused for a 3 minutes")
            #time.sleep(180)
            countdown(180)
            # Code will continue execution after the 3-minute pause
            print("Resumed execution after 3 minutes.")
        j+=1
except:        
    play_music()

https://api.fda.gov/drug/label.json?search=openfda.generic_name:"oxcarbazepine er 600 mg tablet"&limit=1
https://api.fda.gov/drug/label.json?search=openfda.brand_name:"oxcarbazepine er 600 mg tablet"&limit=1
None
https://api.fda.gov/drug/label.json?search=openfda.generic_name:"clozapine 100 mg oral tablet"&limit=1
https://api.fda.gov/drug/label.json?search=openfda.brand_name:"clozapine 100 mg oral tablet"&limit=1
None
https://api.fda.gov/drug/label.json?search=openfda.generic_name:"zoloft"&limit=1
https://api.fda.gov/drug/label.json?search=openfda.brand_name:"zoloft"&limit=1
None
https://api.fda.gov/drug/label.json?search=openfda.generic_name:"clonazepam 0.5mg tablets"&limit=1
https://api.fda.gov/drug/label.json?search=openfda.brand_name:"clonazepam 0.5mg tablets"&limit=1
None
https://api.fda.gov/drug/label.json?search=openfda.generic_name:"clonazepam   tab 2mg"&limit=1
https://api.fda.gov/drug/label.json?search=openfda.brand_name:"clonazepam   tab 2mg"&limit=1
None
https://api.fda.gov

In [246]:
#med_class_se_dict
df['SE_med_nm'] = df['medication_name'].map(med_class_se_dict)

In [247]:
# Get side effects from Medication Name. Create a dictionary mapping the RXCUI to the class

# Create dictionary 
ing_class_se_dict = {}

# Extract distinct med_names's & Not nulls
ing_nm_keys = list(set(sorted(df[df['ingredient_name'].notna()]['ingredient_name'])))
j = 0

#print(keys)
#Print class from the rxcui and insert into the dictionary
try:
    for i in range(0,len(ing_nm_keys)):
        #print(keys[i])
        print('%d of %d'%(j,len(ing_nm_keys)),end='\r')
        combined_text = get_openfda_ae_terms(ing_nm_keys[i])
        #print(combined_text)
        if combined_text != None:
            ing_class_se_dict[ing_nm_keys[i]] = extract_side_effects_from_text(sider_terms,' '.join(combined_text))
        #print()
        if (j>0 and j%100 == 0):
            print("Execution paused for a 3 minutes")
            #time.sleep(180)
            countdown(180)
            # Code will continue execution after the 3-minute pause
            print("Resumed execution after 3 minutes.")
            print("Started at %d"%j)
        j+=1
except:        
    play_music()

Execution paused for a 3 minutes
Resumed execution after 3 minutes.
Started at 100
117 of 118

In [248]:
df['SE_ing_nm'] = df['ingredient_name'].map(ing_class_se_dict)

In [249]:
df.to_csv('test_2.csv')

In [250]:
def make_merge_unique_words_function(cols_to_merge):
    def merge_unique_words(row):
        words = set()
        for col in cols_to_merge:
            val = row[col]
            if isinstance(val, str):
                words.update(val.lower().split())
        return ' '.join(sorted(words))
    return merge_unique_words

In [251]:
cols = ['SE_rxcui', 'SE_med_nm', 'SE_ing_nm']
merge_func = make_merge_unique_words_function(cols)
df['Side_Effects_Combined'] = df.apply(merge_func, axis=1)

In [253]:
df.head(5)

,Unnamed: 0,rxcui,medication_name,dose,ingredient_name,indication_terms_fda,coalesced_class_name,SE_rxcui,SE_med_nm,SE_ing_nm,Side_Effects_Combined
0,0,1000048.0,doxepin 10 mg oral capsule,10 mg,doxepin,"[('insomnia', 2)]",Non-selective monoamine reuptake inhibitors (A...,"aggressiveness, agitation, agranulocytosis, ak...",NaN,"abdominal pain, abnormal dreams, accident, ade...","abdominal abnormal abnormal, accident, adenoca..."
1,1,1000048.0,doxepin 10 mg oral capsule,20 mg,doxepin,"[('insomnia', 2)]",Non-selective monoamine reuptake inhibitors (A...,"aggressiveness, agitation, agranulocytosis, ak...",NaN,"abdominal pain, abnormal dreams, accident, ade...","abdominal abnormal abnormal, accident, adenoca..."
2,2,1000048.0,doxepin 10 mg oral capsule,30 mg,doxepin,"[('insomnia', 2)]",Non-selective monoamine reuptake inhibitors (A...,"aggressiveness, agitation, agranulocytosis, ak...",NaN,"abdominal pain, abnormal dreams, accident, ade...","abdominal abnormal abnormal, accident, adenoca..."
3,3,1000048.0,doxepin 10 mg oral capsule,NaN,doxepin,"[('insomnia', 2)]",Non-selective monoamine reuptake inhibitors (A...,"aggressiveness, agitation, agranulocytosis, ak...",NaN,"abdominal pain, abnormal dreams, accident, ade...","abdominal abnormal abnormal, accident, adenoca..."
4,4,1000048.0,doxepin hydrochloride 10 mg oral capsule,NaN,doxepin,"[('insomnia', 2)]",Non-selective monoamine reuptake inhibitors (A...,"aggressiveness, agitation, agranulocytosis, ak...",NaN,"abdominal pain, abnormal dreams, accident, ade...","abdominal abnormal abnormal, accident, adenoca..."


In [254]:
df_2 = df[['rxcui','medication_name','dose','ingredient_name','indication_terms_fda','coalesced_class_name','Side_Effects_Combined']]

In [255]:
df_2

,rxcui,medication_name,dose,ingredient_name,indication_terms_fda,coalesced_class_name,Side_Effects_Combined
0,1000048.0,doxepin 10 mg oral capsule,10 mg,doxepin,"[('insomnia', 2)]",Non-selective monoamine reuptake inhibitors (A...,"abdominal abnormal abnormal, accident, adenoca..."
1,1000048.0,doxepin 10 mg oral capsule,20 mg,doxepin,"[('insomnia', 2)]",Non-selective monoamine reuptake inhibitors (A...,"abdominal abnormal abnormal, accident, adenoca..."
2,1000048.0,doxepin 10 mg oral capsule,30 mg,doxepin,"[('insomnia', 2)]",Non-selective monoamine reuptake inhibitors (A...,"abdominal abnormal abnormal, accident, adenoca..."
3,1000048.0,doxepin 10 mg oral capsule,NaN,doxepin,"[('insomnia', 2)]",Non-selective monoamine reuptake inhibitors (A...,"abdominal abnormal abnormal, accident, adenoca..."
4,1000048.0,doxepin hydrochloride 10 mg oral capsule,NaN,doxepin,"[('insomnia', 2)]",Non-selective monoamine reuptake inhibitors (A...,"abdominal abnormal abnormal, accident, adenoca..."
...,...,...,...,...,...,...,...
967,NaN,vilazodone hcl 40 mg tablet,NaN,NaN,NaN,Other antidepressants (ATC1-4),
968,NaN,vilazodone tab 10mg,NaN,NaN,NaN,Other antidepressants (ATC1-4),
969,NaN,zolpidem er 12.5mg tablets,NaN,NaN,NaN,gamma-Aminobutyric Acid A Receptor Positive Mo...,
970,NaN,zolpidem tart er 12.5 mg tab,NaN,NaN,NaN,gamma-Aminobutyric Acid A Receptor Positive Mo...,


In [267]:
#Rough

text = "drowsiness, fatigue, muscle weakness, ataxia, confusion, depression, dysarthria, headache, slurred speech, tremor, vertigo Gastrointestinal System : constipation, nausea, gastrointestinal disturbances, blurred vision, diplopia, dizziness, hypotension, stimulation, restlessness, acute hyperexcited states, anxiety, agitation, aggressiveness, irritability, rage, hallucinations, psychoses, delusions, increased muscle spasticity, insomnia, sleep disturbances, nightmares, Inappropriate behavior, other adverse behavioral effects, incontinence, 'changes in libido', urinary retention Skin, Appendages, elevated transaminase, alkaline phosphatase, 'changes in salivation', including dry mouth, hypersalivation Antegrade amnesia, 'Minor changes in EEG patterns', usually low-voltage fast activity, 'falls and fractures', 'Injury, Poisoning and Procedural Complications'"

In [268]:
result = text.split(',')

In [269]:
result

['drowsiness',
 ' fatigue',
 ' muscle weakness',
 ' ataxia',
 ' confusion',
 ' depression',
 ' dysarthria',
 ' headache',
 ' slurred speech',
 ' tremor',
 ' vertigo Gastrointestinal System : constipation',
 ' nausea',
 ' gastrointestinal disturbances',
 ' blurred vision',
 ' diplopia',
 ' dizziness',
 ' hypotension',
 ' stimulation',
 ' restlessness',
 ' acute hyperexcited states',
 ' anxiety',
 ' agitation',
 ' aggressiveness',
 ' irritability',
 ' rage',
 ' hallucinations',
 ' psychoses',
 ' delusions',
 ' increased muscle spasticity',
 ' insomnia',
 ' sleep disturbances',
 ' nightmares',
 ' Inappropriate behavior',
 ' other adverse behavioral effects',
 ' incontinence',
 " 'changes in libido'",
 ' urinary retention Skin',
 ' Appendages',
 ' elevated transaminase',
 ' alkaline phosphatase',
 " 'changes in salivation'",
 ' including dry mouth',
 ' hypersalivation Antegrade amnesia',
 " 'Minor changes in EEG patterns'",
 ' usually low-voltage fast activity',
 " 'falls and fractures'",


In [274]:
for i in result:
    print('----')
    print(i.lstrip())

----
drowsiness
----
fatigue
----
muscle weakness
----
ataxia
----
confusion
----
depression
----
dysarthria
----
headache
----
slurred speech
----
tremor
----
vertigo Gastrointestinal System : constipation
----
nausea
----
gastrointestinal disturbances
----
blurred vision
----
diplopia
----
dizziness
----
hypotension
----
stimulation
----
restlessness
----
acute hyperexcited states
----
anxiety
----
agitation
----
aggressiveness
----
irritability
----
rage
----
hallucinations
----
psychoses
----
delusions
----
increased muscle spasticity
----
insomnia
----
sleep disturbances
----
nightmares
----
Inappropriate behavior
----
other adverse behavioral effects
----
incontinence
----
'changes in libido'
----
urinary retention Skin
----
Appendages
----
elevated transaminase
----
alkaline phosphatase
----
'changes in salivation'
----
including dry mouth
----
hypersalivation Antegrade amnesia
----
'Minor changes in EEG patterns'
----
usually low-voltage fast activity
----
'falls and fractures'